In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
MatchVotesInfo_df = pd.read_parquet(zip_folder / "votes_all.parquet")

MatchEventInfo_df = pd.read_parquet(zip_folder / "event_all.parquet")

# From MatchEventInfo, we only want the columns winner_code and match_id to enable merging!
# Then we drop duplicates and NaN values!
MatchEventInfo_df = MatchEventInfo_df[["match_id","winner_code"]]
MatchEventInfo_df.drop_duplicates(inplace=True)
MatchEventInfo_df.dropna(subset="winner_code", inplace=True)

# In MatchVotes, we drop date column to enable dropping duplicates
# The tables are actually snapshots of data, so it is convenient to keep the last data.
MatchVotesInfo_df.drop(columns="date",inplace=True)
MatchVotesInfo_df.drop_duplicates("match_id",keep="last",inplace=True)

# Now, it is time to merge data
# We use inner join to avoid creating NaN values!
result_df = pd.merge(MatchEventInfo_df, MatchVotesInfo_df, on="match_id", how="inner")

In [2]:
# Now, let's create some useful columns
result_df["total_votes"] = (result_df["home_vote"] + result_df["away_vote"])

result_df["winner_votes"] = np.where( result_df["winner_code"] == 1, result_df["home_vote"], result_df["away_vote"])

result_df["loser_votes"] = np.where(result_df["winner_code"] == 1,result_df["away_vote"],result_df["home_vote"])

result_df["winner_fan_accuracy"] = (result_df["winner_votes"] /result_df["total_votes"] * 100)

result_df["loser_fan_accuracy"] = (result_df["loser_votes"] /result_df["total_votes"] * 100)

# Now, drop NaN values from accuracy columns and filter data
result_df.dropna(subset=["winner_fan_accuracy","loser_fan_accuracy"], inplace=True)
result_df.sort_values(by="winner_fan_accuracy", ascending=False, inplace=True)
result_df = result_df[(result_df["home_vote"] >= 20) & (result_df["away_vote"] >= 20)]

# Show result:
result_df.head(20)

,match_id,winner_code,home_vote,away_vote,total_votes,winner_votes,loser_votes,winner_fan_accuracy,loser_fan_accuracy
7162,12087305,1.0,2502,40,2542,2502,40,98.426436,1.573564
7175,12087942,2.0,116,6438,6554,6438,116,98.230088,1.769912
9380,12109963,2.0,142,5562,5704,5562,142,97.510519,2.489481
7879,12122309,1.0,1114,30,1144,1114,30,97.377622,2.622378
10067,12147249,1.0,752,22,774,752,22,97.157623,2.842377
7168,12087900,1.0,6228,184,6412,6228,184,97.130381,2.869619
7889,12122324,2.0,20,542,562,542,20,96.441281,3.558719
9381,12109978,1.0,4918,218,5136,4918,218,95.755452,4.244548
7159,12087270,1.0,1608,72,1680,1608,72,95.714286,4.285714
7165,12087318,2.0,130,2774,2904,2774,130,95.523416,4.476584
